# Encroachment Building Segmenter — Kaggle 2× GPU Training

Trains the YOLO segmentation model consumed by `backend/services/encroachment/` (see `backend/services/encroachment/segmenter.py`) on the Roboflow [`encroachment-dwic8`](https://universe.roboflow.com/encroachment/encroachment-dwic8/dataset/31) dataset (v31, single class `Buildings`).

**Kaggle setup**

1. New notebook → **Settings → Accelerator → `GPU T4 x2`**.
2. **Settings → Internet → On** (needed for the Roboflow download).
3. **Add-ons → Secrets → Add new secret** named `ROBOFLOW_API_KEY` with your Roboflow account key.
4. Run all cells (≈ 60–90 min for 80 epochs on 2× T4).

Trained weights are written to `/kaggle/working/best_encroachment.pt`. Download from the right-hand **Output** panel and drop the file at the repo root, or point `AI_ENCROACHMENT_MODEL_PATH` in `backend/.env` at it.

## 1. Install dependencies

In [ ]:
%pip install -q -U ultralytics==8.3.40 roboflow==1.1.50 pyyaml

## 2. Verify the GPU accelerator

If `device_count` is less than 2 the notebook is not on the `GPU T4 x2` accelerator — switch it before training.

In [ ]:
import subprocess
import torch

print(subprocess.check_output(["nvidia-smi", "-L"]).decode().strip())
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
assert torch.cuda.is_available(), "No CUDA device — enable GPU T4 x2 in Kaggle settings."
assert torch.cuda.device_count() >= 2, "Select 'GPU T4 x2' so DDP can use both GPUs."

## 3. Pull the Roboflow dataset

The Kaggle Secrets Manager exposes `ROBOFLOW_API_KEY` to the notebook. Replace the `UserSecretsClient` call below with a literal string if you'd rather paste the key inline (do not commit it).

In [ ]:
import os
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    ROBOFLOW_API_KEY = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
except Exception:
    ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "")

assert ROBOFLOW_API_KEY, "Set ROBOFLOW_API_KEY in Kaggle Secrets (or env) before running this cell."

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("encroachment").project("encroachment-dwic8")
dataset = project.version(31).download("yolov8", location="/kaggle/working/encroachment")
DATASET_DIR = Path(dataset.location).resolve()
DATA_YAML = DATASET_DIR / "data.yaml"
print("Dataset location:", DATASET_DIR)
print("data.yaml:", DATA_YAML)

## 4. Pin absolute paths in `data.yaml`

Roboflow exports a relative-path yaml; rewriting it with `path:` plus train/val/test subdirs is the safest way to keep DDP workers happy regardless of cwd.

In [ ]:
import yaml

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

cfg["path"] = str(DATASET_DIR)
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
cfg["test"] = "test/images"

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(yaml.safe_dump(cfg, sort_keys=False))
for split in ("train", "valid", "test"):
    img_dir = DATASET_DIR / split / "images"
    print(f"{split:6s} {sum(1 for _ in img_dir.glob('*')):>5d} images at {img_dir}")

## 5. Train YOLOv8n-seg across both GPUs

`device=[0, 1]` enables Ultralytics' DDP path; `batch` is the **global** mini-batch size (so 32 = 16 per GPU). Bump to `yolov8s-seg.pt` / increase epochs if val mAP keeps climbing.


In [ ]:
from ultralytics import YOLO

os.environ["WANDB_DISABLED"] = "true"

model = YOLO("yolov8n-seg.pt")
results = model.train(
    data=str(DATA_YAML),
    task="segment",
    epochs=80,
    imgsz=640,
    batch=32,
    device=[0, 1],
    workers=2,
    optimizer="AdamW",
    lr0=1e-3,
    cos_lr=True,
    patience=20,
    seed=42,
    project="/kaggle/working/runs",
    name="encroachment",
    exist_ok=True,
    plots=True,
)
print("Best weights:", results.save_dir if hasattr(results, 'save_dir') else "/kaggle/working/runs/encroachment")

## 6. Validate on the held-out split

In [ ]:
BEST = Path("/kaggle/working/runs/encroachment/weights/best.pt")
assert BEST.exists(), f"Expected weights at {BEST}; check the training cell output."

val_model = YOLO(str(BEST))
metrics = val_model.val(
    data=str(DATA_YAML),
    device=0,
    imgsz=640,
    plots=True,
)
print(f"box mAP50-95: {metrics.box.map:.4f}")
print(f"seg mAP50-95: {metrics.seg.map:.4f}")

## 7. Save weights to `/kaggle/working`

Anything under `/kaggle/working` shows up as a downloadable output artifact.

In [ ]:
import shutil

OUT = Path("/kaggle/working/best_encroachment.pt")
shutil.copy2(BEST, OUT)
print(f"Saved {OUT} ({OUT.stat().st_size / 1e6:.1f} MB)")

## 8. Spot-check on a few validation tiles

In [ ]:
import matplotlib.pyplot as plt

samples = sorted((DATASET_DIR / "valid" / "images").glob("*.jpg"))[:3]
predict_model = YOLO(str(OUT))
for img_path in samples:
    preds = predict_model.predict(source=str(img_path), conf=0.25, device=0, verbose=False)
    if not preds:
        continue
    annotated = preds[0].plot()
    plt.figure(figsize=(8, 6))
    plt.imshow(annotated[:, :, ::-1])
    plt.axis("off")
    plt.title(img_path.name)
    plt.show()

## 9. Use the weights in the app

From the Kaggle UI's **Output** panel, download `best_encroachment.pt`. Two ways to wire it up:

- Copy the file to the repo root next to `best_floor.pt` (the default `AI_ENCROACHMENT_MODEL_PATH=../best_encroachment.pt` already points there), or
- Put it anywhere and set `AI_ENCROACHMENT_MODEL_PATH=/absolute/path/to/best_encroachment.pt` in `backend/.env`.

Restart the FastAPI server and submit an aerial / satellite image with GPS — the encroachment pipeline will pick the weights up automatically and return the per-category area breakdown.

Tune in `backend/.env` if the screenshot scale differs from the citizen submissions you receive:

- `AI_ENCROACHMENT_CONFIDENCE` — YOLO confidence threshold (default `0.25`).
- `AI_ENCROACHMENT_IMAGE_SPAN_M` — real-world span of the longer image side (default `300.0` m).
- `AI_ENCROACHMENT_ROAD_BUFFER_M` — half-width applied to OSM road centerlines (default `4.0` m).